# 02 -- Build 1-min feature dataset

Canonical 1-min-cadence pipeline. Reads `data/raw_data/klines_1m.parquet`, runs the feature engine at `label_cadence='1min'`, and writes the model dataset along with its feature list and metadata sidecar.

Outputs:
- `data/model_dataset/dataset_1min.parquet`
- `data/model_dataset/feature_list_1min.json`
- `data/model_dataset/dataset_metadata_1min.json`

This notebook is the only active feature-building entrypoint. The pre-refactor boundary-cadence pipeline is archived under `legacy/` and produces no active artifacts.

In [1]:
from __future__ import annotations

import json
import os
import sys
import time
import warnings
from pathlib import Path

import polars as pl
import pandas as pd

# Resolve repo root robustly (works whether you launch Jupyter from repo root or notebooks/).
ROOT = Path.cwd()
if not (ROOT / 'docs' / 'MINIMAL_PROJECT_SPEC_v2.md').exists():
    if (ROOT.parent / 'docs' / 'MINIMAL_PROJECT_SPEC_v2.md').exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError('Could not locate repo root.')
sys.path.insert(0, str(ROOT))

from src.analytics.audits import causal_feature_audit
from src.features.config import C, ETA, K_WARMUP, M, N_WARMUP, PHI
from src.features.pipeline import (
    _BASE_COLS,
    _DERIV_BASE_COLS,
    _LABEL_AUX_COLS,
    _RAW_COLS,
    run_pipeline,
)
from src.utils import get_git_sha

warnings.filterwarnings('ignore')
os.environ.setdefault('PYTHONIOENCODING', 'utf-8')

# Default target/execution alignment: at 1-min cadence the production
# strategy fills a long TP on intrabar HIGH crossing, so the label tests
# future highs (default in run_pipeline). The build records the source
# used so a model train + strategy backtest cannot silently drift apart.
# Flip to "close" for the legacy close-confirmed variant.
BARRIER_SOURCE = "high"

RAW_PATH = ROOT / 'data' / 'raw_data' / 'klines_1m.parquet'
OUT_DIR = ROOT / 'data' / 'model_dataset'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('RAW_PATH:', RAW_PATH)
print('OUT_DIR:', OUT_DIR)
print('BARRIER_SOURCE:', BARRIER_SOURCE)


ROOT: C:\Users\vitil\OneDrive\Desktop\barrier_classifier
RAW_PATH: C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\raw_data\klines_1m.parquet
OUT_DIR: C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset
BARRIER_SOURCE: high


## 1. Load raw 1-min klines

In [2]:
raw = pd.read_parquet(RAW_PATH)
print(f"Raw bars: {len(raw):,} rows  range {raw.index.min()} -> {raw.index.max()}")


Raw bars: 525,600 rows  range 2025-01-01 00:01:00+00:00 -> 2026-01-01 00:00:00+00:00


## 2. Run the 1-min feature pipeline

Calls `run_pipeline(label_cadence='1min', barrier_source=BARRIER_SOURCE, with_derivatives=False)`. The 1-min cadence emits a label on every row (no boundary sparsity), tests intrabar highs by default to match the production strategy's TP fill on `high >= entry * (1 + PHI)`, and produces the canonical feature set without derivatives base series.

In [3]:
print(
    f"Building dataset at label_cadence='1min', barrier_source='{BARRIER_SOURCE}'..."
)
t0 = time.perf_counter()
ds_1min = run_pipeline(
    raw,
    with_derivatives=False,
    label_cadence="1min",
    barrier_source=BARRIER_SOURCE,
)
dt = time.perf_counter() - t0
print(f"  ran in {dt:.1f}s")
print(f"  rows={len(ds_1min):,}  cols={len(ds_1min.columns)}")
print(f"  base_rate y = {float(ds_1min['y'].mean()):.4f}")
print(f"  ts range: {ds_1min['ts'].min()} -> {ds_1min['ts'].max()}")


Building dataset at label_cadence='1min', barrier_source='high'...


  ran in 259.8s
  rows=505,421  cols=1620
  base_rate y = 0.2100
  ts range: 2025-01-15 00:00:00 -> 2025-12-31 23:40:00


## 3. Sanity checks

Counts autocorr / rolling-excursion columns and verifies no boundary-sparse excursion columns leaked through (hard correctness gate).

In [4]:
n_autocorr = sum(1 for c in ds_1min.columns if c.startswith("target__autocorr_"))
print(f"  autocorr cols: {n_autocorr}")
n_roll_excursion = sum(
    1 for c in ds_1min.columns
    if c.startswith("excursion__roll_max_drawup__f__")
    or c.startswith("excursion__roll_max_drawdown__f__")
)
print(f"  rolling-excursion cols (every-row trailing): {n_roll_excursion}")
n_sparse_excursion = sum(
    1 for c in ds_1min.columns
    if c.startswith("excursion__max_drawup__f__")
    or c.startswith("excursion__max_drawdown__f__")
)
# Runtime check (not a Python ``assert``): under ``python -O`` asserts
# are stripped, but this leak invariant is a hard correctness gate.
if n_sparse_excursion != 0:
    raise RuntimeError(
        f"Boundary-sparse excursion cols leaked into the 1-min dataset: "
        f"{n_sparse_excursion}. run_pipeline(label_cadence='1min') should "
        "have dropped these after the engine ran -- check pipeline.py."
    )
print(f"  boundary-sparse excursion cols (must be 0): {n_sparse_excursion}")


  autocorr cols: 16
  rolling-excursion cols (every-row trailing): 26
  boundary-sparse excursion cols (must be 0): 0


## 4. Asymmetric loss weighting (barrier-distance)

Materialize the legacy barrier-distance weight on the training-eligible rows.
Near-miss negatives (`m_k` just below `phi`) are exponentially upweighted so the
classifier learns the geometry of the decision boundary. Positives keep
`weight = 1` (`use_pos=False`); time discount is disabled. The weight is stored
on the dataset so `03_train_model` can pass it to the CatBoost Pool, and is
explicitly excluded from `feature_list` in the next section.

In [5]:
import numpy as np

from src.utils import (
    WEIGHT_DIST_Q_TAIL,
    WEIGHT_DIST_W_MAX,
    compute_training_weights,
)

m_k = ds_1min["m_k"].to_numpy()
phi_arr = ds_1min["phi"].to_numpy()
phi_const = float(phi_arr[0])
if not np.allclose(phi_arr, phi_const):
    raise RuntimeError(
        f"phi is not constant across the dataset; range "
        f"[{float(np.nanmin(phi_arr))}, {float(np.nanmax(phi_arr))}]. "
        "Asymmetric weights assume a single phi."
    )

w_combined, w_dist, w_time, w_info = compute_training_weights(
    m_k=m_k,
    phi=phi_const,
    use_dist=True,
    use_time=False,
    w_max=WEIGHT_DIST_W_MAX,
    q_tail=WEIGHT_DIST_Q_TAIL,
    use_pos=False,
    normalize=False,
)

ds_1min = ds_1min.with_columns(
    pl.Series("weight", w_combined.astype(np.float64))
)

dist_info = w_info["barrier_distance"]
combined_info = w_info["combined"]
print("Asymmetric loss weighting (barrier-distance):")
print(f"  use_dist     : True")
print(f"  use_time     : False")
print(f"  use_pos      : False  (positives keep w=1)")
print(f"  w_max (neg)  : {WEIGHT_DIST_W_MAX}")
print(f"  q_tail (neg) : {WEIGHT_DIST_Q_TAIL}")
print(f"  d_star (neg) : {dist_info['d_star']:.6f}")
print(f"  lambda (neg) : {dist_info['lambda']:.4f}")
print(f"  n positives  : {dist_info['n_positive']:,}")
print(f"  n negatives  : {dist_info['n_negative']:,}")
print(f"  weight range : [{float(w_combined.min()):.4f}, {float(w_combined.max()):.4f}]")
print(f"  weight mean  : {float(w_combined.mean()):.4f}")
print(f"  effective n  : {combined_info['effective_n']:,.1f}  (raw n={len(w_combined):,})")


Asymmetric loss weighting (barrier-distance):
  use_dist     : True
  use_time     : False
  use_pos      : False  (positives keep w=1)
  w_max (neg)  : 5.0
  q_tail (neg) : 0.001
  d_star (neg) : 0.002500
  lambda (neg) : 643.7446
  n positives  : 106,149
  n negatives  : 399,272
  weight range : [1.0000, 5.0000]
  weight mean  : 2.6381
  effective n  : 397,026.9  (raw n=505,421)


## 5. Causal feature audit

Derives the feature list by excluding label aux, raw OHLCV, base series, derivatives base, the asymmetric `weight` column from Section 4, and `undef__*` columns -- reusing the pipeline's authoritative constants so the exclusion set cannot drift from the imputation-step contract. `causal_feature_audit` is a static naming check requiring every feature to contain `__f__` or `__h__` and forbidding suspect tokens like `fwd`, `future`, `ahead`.

In [6]:
# Feature list: every column that is not a label, raw OHLCV, base
# series, or derivatives base series. Reuses the pipeline's
# authoritative constants so the exclusion set cannot drift from the
# imputation-step contract. Triple-barrier aux columns (m_dn, tau_dn)
# are included in _LABEL_AUX_COLS -- sidecar diagnostics, not features.
# The asymmetric `weight` column (Section 4) is consumed by 03_train_model
# via the CatBoost Pool and must not leak into the feature matrix.
NON_FEATURE = (
    set(_LABEL_AUX_COLS)
    | set(_RAW_COLS)
    | set(_BASE_COLS)
    | set(_DERIV_BASE_COLS)
    | {"weight"}
)
feature_cols = [
    c for c in ds_1min.columns
    if c not in NON_FEATURE and not c.startswith("undef__")
]
# Static causal-naming audit: every feature must contain ``__f__`` or
# ``__h__`` and no suspect tokens (``fwd``, ``future``, ``ahead``, ...).
audit = causal_feature_audit(feature_cols)
if not audit.passed:
    raise RuntimeError(
        "causal_feature_audit failed on 1-min build:\n"
        f"  suspect = {audit.suspect[:10]}\n"
        f"  unmatched = {audit.unmatched[:10]}"
    )
if "weight" in feature_cols:
    raise RuntimeError(
        "Asymmetric `weight` column leaked into feature_cols; "
        "fix the NON_FEATURE exclusion set above."
    )
print(f"  causal audit passed: {audit.n_causal}/{audit.n_features} causal")


  causal audit passed: 1511/1511 causal


## 5b. Feature-prep health audit

Static causal-naming is necessary but not sufficient. After it passes, run the post-impute health pass on the materialized feature matrix. The pipeline's impute layer hard-guarantees zero `null` / `NaN` / `inf`, so any non-finite cell surfaced here is a code bug, not a data issue. The interesting signals are:

- **`undef_rate`**: fraction of rows where the feature was warmup-imputed. Large values flag features whose effective warmup exceeds the pipeline's `N_WARMUP` trim — at 1-min cadence `N_WARMUP = 20,159` bars, so any feature with `warmup_for(w) <= 20159` should show `undef_rate == 0`.
- **`null_pattern`**: `leading` / `trailing` / `edge` / `scattered`. Rolling-window warmup produces `leading`; forward-looking labels produce `trailing`. `scattered` is always a bug.
- **`is_imputed_constant`**: feature is constant AND >50% of rows are imputed — model can't learn from it.
- **`is_extreme_outlier`**: `max|x| / median|x|` > 10⁴; flags denominator-near-zero blowups.

If any hard contract fails, the cell raises so we don't silently write a broken dataset.

In [7]:
from src.features.observability import (
    compute_feature_health,
    flag_issues,
    summarize_by_family,
)

n_rows = len(ds_1min)
print(f"Computing per-feature health on {len(feature_cols)} columns x {n_rows:,} rows...")
t0 = time.perf_counter()
health = compute_feature_health(ds_1min, feature_cols, include_stationarity=False)
print(f"  done in {time.perf_counter() - t0:.1f}s")

# Hard contract: every feature must be finite post-impute.
n_null_violations = int(health.filter(pl.col("n_null") > 0).height)
n_nan_violations = int(health.filter(pl.col("n_nan") > 0).height)
n_inf_violations = int(health.filter(pl.col("has_inf")).height)
n_scattered = int(health.filter(pl.col("null_pattern") == "scattered").height)
n_all_missing = int(health.filter(pl.col("null_pattern") == "all_missing").height)

print(f"  residual nulls : {n_null_violations} columns")
print(f"  residual NaNs  : {n_nan_violations} columns")
print(f"  inf cells      : {n_inf_violations} columns")
print(f"  scattered      : {n_scattered} columns")
print(f"  all-missing    : {n_all_missing} columns")
if any([n_null_violations, n_nan_violations, n_inf_violations, n_scattered]):
    raise RuntimeError(
        "Hard correctness gate failed - dataset NOT safe to train. "
        "See the violating columns above and fix the pipeline before re-running."
    )

# eq family contract: max declared warmup is 961 (pair features at L=960
# + 1 for r[0]-null). Pipeline trim is N_WARMUP=20159, so every eq
# column should have undef_rate == 0. Anything > 0 means an
# under-declared warmup_for() somewhere in src/features/families/equilibrium.py.
eq_health = health.filter(pl.col("family") == "eq")
eq_undef_max = float(eq_health["undef_rate"].max() or 0.0)
print(f"  eq family: {len(eq_health)} columns, max undef_rate = {eq_undef_max:.6f}")
if eq_undef_max > 0:
    print("  WARNING: eq has nonzero undef_rate -- inspect warmup_for() declarations")


Computing per-feature health on 1511 columns x 505,421 rows...


  done in 5.8s
  residual nulls : 0 columns
  residual NaNs  : 0 columns
  inf cells      : 0 columns
  scattered      : 0 columns
  all-missing    : 0 columns
  eq family: 123 columns, max undef_rate = 0.000000


### Family-level summary

Per-family aggregates: feature count, average / max undef rate (post-impute), counts of imputed-constant / extreme-outlier / scattered features. Sorted by max undef rate so the families carrying the most warmup pressure surface first.

In [8]:
fam_summary = (
    summarize_by_family(health)
    .sort(["max_undef_rate", "n_features"], descending=[True, True])
)
print(f"Families: {len(fam_summary)}  (total features: {int(fam_summary['n_features'].sum())})")
print()
print(fam_summary.to_pandas().to_string(index=False))


Families: 32  (total features: 1511)

       family  n_features  avg_undef_rate  max_undef_rate  max_missing_rate  n_features_with_nan  n_scattered  n_imputed_constant  n_organic_constant  n_extreme_outlier  n_non_stationary  n_with_inf      avg_std  avg_abs_skew
          ret         336    1.357143e-01        0.950000               0.0                    0            0                   0                   0                  0                 0           0 2.933458e-01      2.549929
       target          44    7.085956e-02        0.341816               0.0                    0            0                   0                   0                  0                 0           0 1.050298e+02      1.683652
        pivot          24    6.842480e-06        0.000164               0.0                    0            0                   0                   0                  0                 0           0 8.755269e+00      1.058884
       absret         102    1.939754e-08        0.000002 

### Warmup and beginning-null audit

Two angles:

1. **Distribution of warmup load**. Histogram-style: how many features have `undef_rate == 0` (warmup fully absorbed by the pipeline trim) vs. various non-zero brackets. Most features should land in the zero bucket.
2. **Top imputed features**. The 20 features with the highest post-impute `undef_rate` and the families they belong to. A new family with `undef_rate > 0` anywhere is the obvious smoke signal for an under-declared `warmup_for`.
3. **eq family deep-dive**. Every `eq__*` column listed alongside its undef rate and dynamic range. The eq family's warmup ladder maxes at 961 rows; the pipeline trims 20,159; so the audit expects zero imputed cells across all 123 eq columns.

In [9]:
import numpy as np

# 1. Distribution of undef_rate.
undef_buckets = [
    ("== 0       ", lambda r: r == 0.0),
    ("(0, 0.001] ", lambda r: 0.0 < r <= 0.001),
    ("(0.001, 0.01]", lambda r: 0.001 < r <= 0.01),
    ("(0.01, 0.1]", lambda r: 0.01 < r <= 0.1),
    ("(0.1, 0.5] ", lambda r: 0.1 < r <= 0.5),
    ("> 0.5      ", lambda r: r > 0.5),
]
rates = np.asarray(health["undef_rate"].to_numpy(), dtype=float)
print("Undef-rate distribution across features:")
for label, predicate in undef_buckets:
    n = int(sum(predicate(r) for r in rates))
    pct = 100.0 * n / len(rates)
    print(f"  {label}: {n:5d}  ({pct:5.1f}%)")
print()

# 2. Top 20 imputed features.
print("Top 20 features by undef_rate:")
print(
    health.sort("undef_rate", descending=True)
    .select(["name", "family", "undef_rate", "null_pattern", "mean", "std"])
    .head(20)
    .to_pandas()
    .to_string(index=False)
)
print()

# 3. eq family deep-dive.
print(f"eq family ({len(eq_health)} columns):")
eq_summary = eq_health.select([
    "name", "undef_rate", "null_pattern", "mean", "std", "min", "max", "outlier_ratio"
]).sort("outlier_ratio", descending=True)
print(eq_summary.to_pandas().to_string(index=False))


Undef-rate distribution across features:
  == 0       :  1434  ( 94.9%)
  (0, 0.001] :     5  (  0.3%)
  (0.001, 0.01]:     6  (  0.4%)
  (0.01, 0.1]:     4  (  0.3%)
  (0.1, 0.5] :    14  (  0.9%)
  > 0.5      :    48  (  3.2%)

Top 20 features by undef_rate:
              name family  undef_rate null_pattern          mean      std
  ret__q10__f__w30    ret        0.95        clean -2.890531e-05 0.000161
  ret__q10__f__w45    ret        0.95        clean -2.956335e-05 0.000162
  ret__q10__f__w60    ret        0.95        clean -2.979511e-05 0.000161
  ret__q10__f__w90    ret        0.95        clean -3.005302e-05 0.000160
 ret__q10__f__w120    ret        0.95        clean -3.018555e-05 0.000158
 ret__q10__f__w180    ret        0.95        clean -3.029063e-05 0.000156
 ret__q10__f__w240    ret        0.95        clean -3.030902e-05 0.000155
 ret__q10__f__w360    ret        0.95        clean -3.032878e-05 0.000153
 ret__q10__f__w720    ret        0.95        clean -3.028712e-05 0.000149

### Issue flags (advisory)

`flag_issues` returns rows that look problematic by any of several heuristics (priority order: residual_nan, inf, scattered_missing, all_missing, extreme_outlier, imputed_constant, non_stationary, organic_constant, high_undef, heavy_skew). The hard-fail categories (residual_nan / inf / scattered) are already gated above; what shows up here is mostly outlier / skew / undef advisories the model trainer should be aware of.

In [10]:
issues = flag_issues(health)
print(f"flag_issues found {len(issues)} flagged features")
issue_counts = issues.group_by("issue").len().sort("len", descending=True)
print()
print("By issue category:")
print(issue_counts.to_pandas().to_string(index=False))
print()
print("By (issue, family):")
print(
    issues.group_by(["issue", "family"]).len()
    .sort(["issue", "len"], descending=[False, True])
    .to_pandas().to_string(index=False)
)
print()
# Spot-check eq specifically: any eq column flagged at all?
eq_issues = issues.filter(pl.col("family") == "eq")
print(f"eq family flagged rows: {len(eq_issues)}")
if len(eq_issues) > 0:
    print(eq_issues.select(["name", "issue", "undef_rate", "outlier_ratio", "skew"]).to_pandas().to_string(index=False))


flag_issues found 68 flagged features

By issue category:
           issue  len
      high_undef   48
      heavy_skew   14
organic_constant    4
 extreme_outlier    2

By (issue, family):
           issue  family  len
 extreme_outlier barrier    1
 extreme_outlier     gap    1
      heavy_skew barrier    6
      heavy_skew extreme    4
      heavy_skew     vol    2
      heavy_skew      eq    2
      high_undef     ret   48
organic_constant    data    2
organic_constant barrier    1
organic_constant    cost    1

eq family flagged rows: 2
                         name      issue  undef_rate  outlier_ratio       skew
eq__median_resid_madz__f__w30 heavy_skew         0.0    7687.190523  74.392305
eq__median_resid_madz__f__w60 heavy_skew         0.0     876.862697 178.550472


## 6. Save artifacts

Writes the parquet dataset (including the asymmetric `weight` column), the feature list JSON, and the metadata sidecar (including the current `git_sha` for provenance).

In [11]:
ds_path = OUT_DIR / 'dataset_1min.parquet'
ds_1min.write_parquet(str(ds_path))
print(f"  wrote {ds_path}")

feature_list_path = OUT_DIR / 'feature_list_1min.json'
with open(feature_list_path, "w") as f:
    json.dump(feature_cols, f, indent=2)
print(f"  feature_list_1min.json: {len(feature_cols)} features")

metadata = {
    "label_cadence": "1min",
    "barrier_source": BARRIER_SOURCE,
    "M": int(M),
    "N_WARMUP": int(N_WARMUP),
    "K_WARMUP": int(K_WARMUP),
    "PHI": float(PHI),
    "C": float(C),
    "ETA": float(ETA),
    "n_rows": int(len(ds_1min)),
    "n_features": len(feature_cols),
    "ts_start": str(ds_1min["ts"].min()),
    "ts_end": str(ds_1min["ts"].max()),
    "base_rate": float(ds_1min["y"].mean()),
    "build_time_seconds": dt,
    "git_sha": get_git_sha(),
    "weighting": {
        "scheme": "barrier_distance_asymmetric",
        "use_dist": True,
        "use_time": False,
        "use_pos": False,
        "w_max_neg": float(WEIGHT_DIST_W_MAX),
        "q_tail_neg": float(WEIGHT_DIST_Q_TAIL),
        "weight_min": float(w_combined.min()),
        "weight_max": float(w_combined.max()),
        "weight_mean": float(w_combined.mean()),
        "effective_n": float(combined_info["effective_n"]),
    },
}
metadata_path = OUT_DIR / 'dataset_metadata_1min.json'
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"  metadata: {metadata}")


  wrote C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset\dataset_1min.parquet
  feature_list_1min.json: 1511 features
  metadata: {'label_cadence': '1min', 'barrier_source': 'high', 'M': 20, 'N_WARMUP': 20159, 'K_WARMUP': 1008, 'PHI': 0.0025, 'C': 0.0023, 'ETA': 0.0002, 'n_rows': 505421, 'n_features': 1511, 'ts_start': '2025-01-15 00:00:00', 'ts_end': '2025-12-31 23:40:00', 'base_rate': 0.2100209528294234, 'build_time_seconds': 259.8204459999688, 'git_sha': '1e1e41ea39291477fc5f17337f996ba582c7a274', 'weighting': {'scheme': 'barrier_distance_asymmetric', 'use_dist': True, 'use_time': False, 'use_pos': False, 'w_max_neg': 5.0, 'q_tail_neg': 0.001, 'weight_min': 1.0, 'weight_max': 4.999999999999999, 'weight_mean': 2.6381032409779084, 'effective_n': 397026.8614957811}}


## 7. Summary

In [12]:
summary = {
    "n_rows": int(len(ds_1min)),
    "n_features": len(feature_cols),
    "base_rate": float(ds_1min["y"].mean()),
    "ts_start": str(ds_1min["ts"].min()),
    "ts_end": str(ds_1min["ts"].max()),
    "barrier_source": BARRIER_SOURCE,
    "build_time_seconds": round(dt, 2),
}
print(summary)


{'n_rows': 505421, 'n_features': 1511, 'base_rate': 0.2100209528294234, 'ts_start': '2025-01-15 00:00:00', 'ts_end': '2025-12-31 23:40:00', 'barrier_source': 'high', 'build_time_seconds': 259.82}
